In [ ]:
import pygmo as pg
import numpy as np
import os
import sys
sys.path.append("/home/jlgpke/projects/UCL/research_project/topological_insulator/notebooks/project/")
from topological_insulator import Problem
from mean_field_problem import MeanFieldProblem

In [ ]:
structure_path = "../../../../../topological_insulator/data/structures/"
structure_name = "honeycomb.json"

In [ ]:
seed = 420

In [ ]:
mean_field_problem = MeanFieldProblem(
    structure_path, structure_name, Delta_SOC=-5,
    t=1, U=3, delta=0.832, occupations= np.zeros(16)
)

mean_field_problem.setup(15, -15, 0.05, T = 300, N_h = 2)

In [ ]:
problem = pg.problem(mean_field_problem)
problem.c_tol = [1e-4]
print(problem)

In [ ]:
inner = pg.de(gen=10) 
inner_algo = pg.algorithm(inner)

In [ ]:
cstrs = pg.cstrs_self_adaptive(15, inner_algo)
algorithm = pg.algorithm(cstrs)
print(algorithm)

In [ ]:
population = pg.population(problem, size=15)

In [ ]:
island = pg.island(algo=algorithm, pop=population)
island.evolve()
island.wait_check()

In [ ]:
champ_x = island.population.champion_x
champ_f = island.population.champion_f
print("Champion x:", champ_x)
print("Objective F:", champ_f[0])
print("Equality constraint diff:", champ_f[1])
print("sum(occ_h):", champ_f[1] + problem.N_h) 

In [ ]:
np.savetxt(f"results/free_energy_dS.txt", champ_x)
np.savetxt(f"results/occupations_dS.txt", champ_f)